In [1]:
from epigenetic_clocks import Horvath, Hannum, PhenoAge, GrimAgeV2, GrimAgeAdapter, run_all_clocks
import pandas as pd
import pickle

dataset = {}
chron_ages = {}
female = {}          # ← now a dict, not a single Series

for data in ['GSE27097', 'GSE40279', 'GSE41037', 'GSE42861']:
    with open(f'/work/gr-fe/bryan/EpiHydra/data/02_processed/{data}/methylation.pkl', 'rb') as file:
        methylation = pickle.load(file)

    dataset[data] = methylation['CpG'].dropna()
    chron_ages[data] = methylation['meta'].set_index('ID')['age']
    female[data] = methylation['meta'].set_index('ID')['gender'] == 'female'   # ← keyed by dataset

clocks = [
    Horvath(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Horvath/horvath.csv"),
    Hannum(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Hannum/hannum.csv", intercept=0.0),
    PhenoAge(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/PhenoAge/phenoage.csv"),
    GrimAgeAdapter(                                                              
        GrimAgeV2(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/GrimAge/GrimAgeV2.csv"),
        chron_ages=chron_ages,
        female=female,
    ),
]

results = run_all_clocks(
    datasets=dataset,
    save_path="/work/gr-fe/bryan/EpiHydra/results",
    clocks=clocks,
    chron_ages=chron_ages,
)

Loaded 353 CpG coefficients + intercept.
Loaded 71 CpG coefficients.
Loaded 513 CpG coefficients + intercept.
Loaded GrimAge V2: 10 sub-components, 1030 unique CpG probes.

  Clocks   : horvath_age, hannum_age, phenoage_age, grimageadapter_age
  Datasets : GSE27097, GSE40279, GSE41037, GSE42861

  Dataset : GSE27097
  Samples : 494   CpGs : 27578

  → Running horvath_age (Horvath) …

  → Running hannum_age (Hannum) …
⚠  64 / 71 clock CpGs (90.1%) not found in methylation data.

  → Running phenoage_age (PhenoAge) …

  → Running grimageadapter_age (GrimAgeAdapter) …
  → Predicting DNAmGDF15 …
⚠  [DNAmGDF15] 126 / 137 CpGs (92.0%) missing from methylation data.
  → Predicting DNAmB2M …
⚠  [DNAmB2M] 84 / 91 CpGs (92.3%) missing from methylation data.
  → Predicting DNAmCystatinC …
⚠  [DNAmCystatinC] 84 / 87 CpGs (96.6%) missing from methylation data.
  → Predicting DNAmTIMP1 …
⚠  [DNAmTIMP1] 40 / 42 CpGs (95.2%) missing from methylation data.
  → Predicting DNAmADM …
⚠  [DNAmADM] 171 / 18

/tmp/40937692/ipykernel_1590899/3568382135.py:19: UserWarning: Hannum intercept is set to 0.0 (the default). This is almost certainly incorrect and will bias every prediction. Supply the correct value from the paper's supplementary materials.
  Hannum(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Hannum/hannum.csv", intercept=0.0),



  → Running hannum_age (Hannum) …

  → Running phenoage_age (PhenoAge) …

  → Running grimageadapter_age (GrimAgeAdapter) …
  → Predicting DNAmGDF15 …
⚠  [DNAmGDF15] 1 / 137 CpGs (0.7%) missing from methylation data.
  → Predicting DNAmB2M …
⚠  [DNAmB2M] 1 / 91 CpGs (1.1%) missing from methylation data.
  → Predicting DNAmCystatinC …
  → Predicting DNAmTIMP1 …
⚠  [DNAmTIMP1] 2 / 42 CpGs (4.8%) missing from methylation data.
  → Predicting DNAmADM …
⚠  [DNAmADM] 23 / 186 CpGs (12.4%) missing from methylation data.
  → Predicting DNAmPAI1 …
⚠  [DNAmPAI1] 11 / 211 CpGs (5.2%) missing from methylation data.
  → Predicting DNAmLeptin …
⚠  [DNAmLeptin] 32 / 187 CpGs (17.1%) missing from methylation data.
  → Predicting DNAmPACKYRS …
⚠  [DNAmPACKYRS] 3 / 172 CpGs (1.7%) missing from methylation data.
  → Predicting DNAmlogCRP …
⚠  [DNAmlogCRP] 11 / 132 CpGs (8.3%) missing from methylation data.
  → Predicting DNAmlogA1C …
⚠  [DNAmlogA1C] 5 / 86 CpGs (5.8%) missing from methylation data.

  A

In [9]:
replace = True

for data in ['GSE27097', 'GSE40279', 'GSE41037', 'GSE42861']:
    
    filepath = f'/work/gr-fe/bryan/EpiHydra/data/02_processed/{data}/methylation.pkl'
    
    with open(filepath, 'rb') as file:
        methylation = pickle.load(file)

    dataset[data] = methylation['CpG']
    chron_ages[data] = methylation['meta'].set_index('ID')['age']

    # Only merge columns that don't already exist in meta
    if replace : 
        overlap = results[data].columns.intersection(methylation['meta'].columns)
        methylation['meta'] = methylation['meta'].drop(columns=overlap)
        
    new_cols = [col for col in results[data].columns if col not in methylation['meta'].columns]
    
    if new_cols:
        methylation['meta'] = pd.merge(
            methylation['meta'],
            results[data][new_cols],
            left_on='ID',
            right_index=True,
            how='left'
        )

    with open(filepath, 'wb') as file:
        pickle.dump(methylation, file)
